# 🎬 VIDEO QUIZ GENERATOR STUDIO — GOOGLE COLAB RUNNER
Notebook này cho phép bạn khởi chạy toàn bộ Studio (Backend + Frontend Remotion) trực tiếp trên Google Colab có GPU/CPU miễn phí.
Chỉ gồm **3 bước (3 Cells)** đơn giản dưới đây:

In [ ]:
# ==============================================================================
# CELL 1: CLONE REPOSITORY TỪ GITHUB (HỖ TRỢ PRIVATE REPO QUA TOKEN)
# ==============================================================================
import os
import shutil

# --- [BƯỚC 1]: ĐIỀN THÔNG TIN REPOSITORY VÀ GITHUB TOKEN CỦA BẠN DƯỚI ĐÂY ---
# Ví dụ: GITHUB_REPO = "username/video-quiz-generator"
GITHUB_REPO = "YOUR_USERNAME/YOUR_REPOSITORY"  # <-- Điền username/repo tại đây

# Tạo token tại: GitHub -> Settings -> Developer settings -> Personal access tokens (classic)
# Quyền (scope) cần thiết: 'repo' (Full control of private repositories)
GITHUB_TOKEN = "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN"  # <-- Điền Personal Access Token tại đây

# ------------------------------------------------------------------------------
REPO_NAME = GITHUB_REPO.split('/')[-1].replace('.git', '') if '/' in GITHUB_REPO else "Video-quiz-new"
WORKSPACE_DIR = f"/content/{REPO_NAME}"

print(f"[*] Đang chuẩn bị clone repository: {GITHUB_REPO}")

if os.path.exists(WORKSPACE_DIR) and os.path.exists(os.path.join(WORKSPACE_DIR, 'package.json')):
    print(f"[✓] Thư mục dự án đã tồn tại tại {WORKSPACE_DIR}. Tiến hành kéo cập nhật mới nhất (git pull)...")
    %cd {WORKSPACE_DIR}
    !git pull origin main || !git pull origin master
else:
    %cd /content
    if GITHUB_TOKEN and GITHUB_TOKEN != "YOUR_GITHUB_PERSONAL_ACCESS_TOKEN":
        AUTH_URL = f"https://{GITHUB_TOKEN}@github.com/{GITHUB_REPO}.git"
    else:
        AUTH_URL = f"https://github.com/{GITHUB_REPO}.git"
    
    !git clone {AUTH_URL} {REPO_NAME}
    %cd {WORKSPACE_DIR}

print(f"\n[✓] Đã chuyển vào thư mục làm việc: {os.getcwd()}")


In [ ]:
# ==============================================================================
# CELL 2: KIỂM TRA & CÀI ĐẶT DEPENDENCY THÔNG MINH (SKIP NẾU ĐÃ CÓ / CÓ CACHE)
# ==============================================================================
import subprocess
import shutil
import os
import sys

def check_command(cmd_name):
    """Kiểm tra một lệnh CLI hệ thống đã tồn tại trong PATH hay chưa."""
    return shutil.which(cmd_name) is not None

def check_python_package(package_name):
    """Kiểm tra một Python package đã được cài đặt hay chưa."""
    try:
        __import__(package_name.replace('-', '_'))
        return True
    except ImportError:
        return False

print("=== KIỂM TRA MÔI TRƯỜNG & DEPENDENCIES ===")

# 1. Kiểm tra Node.js (Yêu cầu Node >= 18 cho Remotion & Vite)
node_ok = False
if check_command("node"):
    try:
        node_ver = subprocess.check_output(["node", "-v"]).decode().strip()
        major = int(node_ver.lstrip('v').split('.')[0])
        if major >= 18:
            print(f"[✓] Node.js đã sẵn sàng: {node_ver}")
            node_ok = True
    except Exception:
        pass

if not node_ok:
    print("[*] Đang cài đặt Node.js v20 LTS...")
    !curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
    !apt-get install -y nodejs > /dev/null 2>&1
    print("[✓] Đã cài đặt Node.js thành công!")

# 2. Kiểm tra FFmpeg
if check_command("ffmpeg"):
    ffmpeg_ver = subprocess.check_output(["ffmpeg", "-version"]).decode().split('\n')[0]
    print(f"[✓] FFmpeg đã sẵn sàng: {ffmpeg_ver}")
else:
    print("[*] Đang cài đặt FFmpeg...")
    !apt-get update -qq && !apt-get install -y -qq ffmpeg > /dev/null 2>&1
    print("[✓] Đã cài đặt FFmpeg thành công.")

# 3. Kiểm tra Python packages (edge-tts)
python_pkgs = ["edge-tts"]
for pkg in python_pkgs:
    if check_python_package(pkg):
        print(f"[✓] Python package '{pkg}' đã sẵn sàng.")
    else:
        print(f"[*] Đang cài đặt Python package '{pkg}'...")
        !pip install -q {pkg}
        print(f"[✓] Đã cài đặt '{pkg}'.")

# 4. Kiểm tra & Cài đặt Node modules (Tận dụng Cache)
if os.path.exists("node_modules") and os.path.exists("node_modules/remotion"):
    print("[✓] Thư mục node_modules đã tồn tại. Bỏ qua npm install để tiết kiệm thời gian.")
else:
    print("[*] Đang cài đặt npm packages (sử dụng cache)... Chờ khoảng 1-2 phút...")
    !npm install --prefer-offline --no-audit --loglevel=error
    print("[✓] npm install hoàn tất!")

# 5. Tải trước Chrome Headless Shell cho Remotion (tối ưu tốc độ render siêu tốc)
print("[*] Đang chuẩn bị Chrome Headless Shell cho Remotion...")
!node -e "import('@remotion/renderer').then(r => r.ensureBrowser({ chromeMode: 'headless-shell' })).then(res => console.log('[✓] Remotion Headless Shell sẵn sàng:', res.path)).catch(e => console.log('[!] Browser:', e.message))"
print("[✓] Môi trường và Dependencies đã sẵn sàng!\n")


In [ ]:
# ==============================================================================
# CELL 3: KHỞI CHẠY STUDIO & TUNNEL FOREGROUND (DUY TRÌ LIÊN TỤC CHO ĐẾN KHI DỪNG)
# ==============================================================================
import subprocess
import time
import urllib.request
import urllib.error
import re
import os
import shutil
from IPython.display import display, HTML

# 1. CẤU HÌNH CỔNG (MẶC ĐỊNH 4500, BACKEND 5410)
FRONTEND_PORT = int(os.environ.get("FRONTEND_PORT", 4500))
BACKEND_PORT = int(os.environ.get("BACKEND_PORT", 5410))
os.environ["FRONTEND_PORT"] = str(FRONTEND_PORT)
os.environ["BACKEND_PORT"] = str(BACKEND_PORT)

print("=" * 70)
print(f"[*] KHỞI CHẠY STUDIO VIDEO TRÊN CỔNG: {FRONTEND_PORT} (Backend: {BACKEND_PORT})")
print("=" * 70)

# Dọn dẹp process cũ trên các cổng
!fuser -k {FRONTEND_PORT}/tcp > /dev/null 2>&1 || true
!fuser -k 5400/tcp > /dev/null 2>&1 || true
!fuser -k {BACKEND_PORT}/tcp > /dev/null 2>&1 || true
!pkill -f cloudflared > /dev/null 2>&1 || true
!pkill -f localtunnel > /dev/null 2>&1 || true

# Khởi động Backend & Frontend ở chế độ background
server_log_path = "/content/studio_server.log"
server_log = open(server_log_path, "w")
server_proc = subprocess.Popen(
    ["npm", "run", "dev"],
    stdout=server_log,
    stderr=subprocess.STDOUT,
    shell=False
)

# 2. CHỜ SERVER THỰC SỰ READY TRƯỚC KHI TẠO TUNNEL
def probe_port(port):
    try:
        with urllib.request.urlopen(f"http://127.0.0.1:{port}", timeout=1.5) as resp:
            if resp.status in [200, 301, 302, 304]:
                return True
    except Exception:
        pass
    return False

print("[*] Đang chờ Web Server khởi động và sẵn sàng (READY)...")
active_port = None
for attempt in range(45):
    time.sleep(2)
    for p in [FRONTEND_PORT, 5400]:
        if probe_port(p):
            active_port = p
            break
    if active_port:
        break
    if (attempt + 1) % 5 == 0:
        print(f"    ... đang kết nối máy chủ ({attempt + 1}/45)")

if not active_port:
    print("[❌] LỖI: Server không phản hồi sau 90 giây. Chi tiết nhật ký log:")
    if os.path.exists(server_log_path):
        with open(server_log_path, "r", errors="ignore") as f:
            print(f.read()[-1500:])
    raise RuntimeError("Server failed to start in time")

print(f"[✓] Server đã READY và phản hồi thành công trên cổng: {active_port}!\n")

# 3. KIỂM TRA & CÀI ĐẶT CLOUDFLARED (CHỈ CÀI KHI THIẾU)
if shutil.which("cloudflared"):
    print("[✓] Cloudflare Tunnel (cloudflared) đã có sẵn trên hệ thống.")
else:
    print("[*] Đang tải và cài đặt cloudflared...")
    !wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
    !dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1
    print("[✓] Đã cài đặt cloudflared thành công.")

# 4. HÀM KIỂM TRA ĐỘ SỐNG CỦA URL TRƯỚC KHI BÁO THÀNH CÔNG
def verify_url(url, timeout=8):
    try:
        req = urllib.request.Request(
            url,
            headers={"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64)"}
        )
        with urllib.request.urlopen(req, timeout=timeout) as resp:
            if resp.status in [200, 301, 302, 304]:
                return True, resp.status
    except urllib.error.HTTPError as e:
        if e.code in [200, 301, 302, 304]:
            return True, e.code
        return True, e.code
    except Exception as e:
        return False, str(e)
    return False, "No response"

# 5. KHỞI TẠO TUNNEL (ƯU TIÊN CLOUDFLARE, FALLBACK LOCALTUNNEL NẾU THẤT BẠI)
def extract_cf_url(log_path):
    if not os.path.exists(log_path):
        return None
    try:
        with open(log_path, "r", errors="ignore") as f:
            c = f.read()
        matches = re.findall(r"https://([a-zA-Z0-9\-]+)\.trycloudflare\.com", c)
        valid = [f"https://{m}.trycloudflare.com" for m in matches if m.lower() != "api"]
        if valid:
            return valid[-1]
    except Exception:
        pass
    return None

public_url = None
tunnel_type = "Cloudflare"
tunnel_proc = None
tunnel_pass = None
cf_log = "/content/cloudflared.log"
!rm -f {cf_log}

print(f"[*] Đang khởi tạo Cloudflare Tunnel cho cổng {active_port} (Giao thức HTTP/2 IPv4 an toàn)...")
cf_cmd = [
    "cloudflared", "tunnel",
    "--protocol", "http2",
    "--edge-ip-version", "4",
    "--url", f"http://127.0.0.1:{active_port}"
]
tunnel_proc = subprocess.Popen(cf_cmd, stdout=open(cf_log, "w"), stderr=subprocess.STDOUT)

# Chờ Cloudflare cấp tên miền trong tối đa 16 giây
for attempt in range(12):
    time.sleep(1.5)
    public_url = extract_cf_url(cf_log)
    if public_url:
        break

# Nếu Cloudflare thất bại, hiển thị nguyên nhân và chuyển ngay sang Localtunnel
if not public_url:
    print("[!] CẢNH BÁO: Cloudflare Tunnel không cấp được URL.")
    if os.path.exists(cf_log):
        with open(cf_log, "r", errors="ignore") as f:
            log_tail = f.read()[-800:]
            print(f"[*] Chi tiết nguyên nhân từ Cloudflare:\n{log_tail.strip()}")
    try:
        tunnel_proc.terminate()
    except Exception:
        pass

    print("\n[*] Đang tự động chuyển sang đường hầm dự phòng: LOCALTUNNEL...")
    try:
        tunnel_pass = urllib.request.urlopen("https://ipv4.icanhazip.com", timeout=4).read().decode().strip()
    except Exception:
        tunnel_pass = ""

    lt_log = "/content/localtunnel.log"
    !rm -f {lt_log}
    lt_cmd = ["npx", "--yes", "localtunnel", "--port", str(active_port)]
    tunnel_proc = subprocess.Popen(lt_cmd, stdout=open(lt_log, "w"), stderr=subprocess.STDOUT)
    tunnel_type = "Localtunnel"

    for _ in range(25):
        time.sleep(1.5)
        if os.path.exists(lt_log):
            with open(lt_log, "r", errors="ignore") as f:
                content = f.read()
                m = re.search(r"https://[a-zA-Z0-9\-]+\.loca\.lt", content)
                if m:
                    public_url = m.group(0)
                    break

# 6. KIỂM TRA TRUY CẬP THỰC TẾ TRƯỚC KHI BÁO THÀNH CÔNG
colab_direct_url = None
try:
    from google.colab.output import eval_js
    colab_direct_url = eval_js(f"google.colab.kernel.proxyPort({active_port})")
except Exception:
    pass

if public_url:
    print(f"[*] Đang kiểm tra độ phản hồi thực tế của URL: {public_url}...")
    time.sleep(2)
    is_ok, check_msg = verify_url(public_url)
    if is_ok:
        print(f"[✓] Đã kiểm tra kết nối tới URL thành công (phản hồi: {check_msg})!")
    else:
        print(f"[!] Cảnh báo: URL vừa tạo nhưng có thể cần thêm vài giây để DNS toàn cầu thông ({check_msg}).")
else:
    print("[!] Không tạo được Public Tunnel bên ngoài. Tự động chuyển dùng Google Colab Proxy.")
    if colab_direct_url:
        public_url = colab_direct_url
        tunnel_type = "Google Colab Direct"

# 7. IN VÀ HIỂN THỊ PUBLIC URL RÕ RÀNG
print("\n" + "=" * 70)
print("🚀 STUDIO VIDEO QUIZ ĐÃ KHỞI CHẠY THÀNH CÔNG & ĐANG SỐNG TRÊN COLAB!")
print("=" * 70)

if public_url:
    print(f"\n👉 PUBLIC URL TRUY CẬP WEB CỦA BẠN ({tunnel_type}):\n   {public_url}\n")

if tunnel_type == "Localtunnel" and tunnel_pass:
    print(f"🔑 Mật khẩu Tunnel IP (nếu trang web yêu cầu): {tunnel_pass}\n")

if colab_direct_url and colab_direct_url != public_url:
    print(f"🔗 Link Google Colab Proxy (Dự phòng ổn định):\n   {colab_direct_url}\n")

try:
    btn_html = f'''
        <div style="background:#0f172a;border:2px solid #00e5ff;border-radius:12px;padding:22px;text-align:center;margin:15px 0;">
            <h3 style="color:#ffffff;margin:0 0 12px 0;">🎬 Studio Video Quiz Đang Chạy</h3>
            <a href="{public_url}" target="_blank" style="display:inline-block;background:#00e5ff;color:#000000;font-weight:bold;font-size:16px;padding:12px 28px;border-radius:8px;text-decoration:none;box-shadow:0 0 15px rgba(0,229,255,0.4);margin:6px;">
                👉 BẤM VÀO ĐÂY ĐỂ MỞ TOOL TRÊN TRÌNH DUYỆT
            </a>
            {f'<p style="color:#fbbf24;font-size:13px;margin:10px 0 0 0;">🔑 Mật khẩu IP (nếu web hỏi): <b>{tunnel_pass}</b></p>' if tunnel_type == "Localtunnel" and tunnel_pass else ''}
        </div>
    '''
    display(HTML(btn_html))
except Exception:
    pass

print("=" * 70)
print("🔒 CELL NÀY ĐANG CHẠY FOREGROUND VÀ DUY TRÌ TIẾN TRÌNH SỐNG LIÊN TỤC.")
print("⚠️  VUI LÒNG KHÔNG BẤM DỪNG (STOP) CELL TRONG KHI BẠN ĐANG DÙNG HOẶC RENDER!")
print("💡 Khi nào bạn muốn tắt hoàn toàn Studio, hãy bấm nút Dừng (Stop) trên Colab.\n")
print("=" * 70 + "\n")

# 8. VÒNG LẶP FOREGROUND DUY TRÌ LIÊN TỤC VÀ TỰ PHỤC HỒI
start_time = time.time()
last_heartbeat = 0
try:
    while True:
        time.sleep(3)
        now = time.time()
        elapsed = int(now - start_time)

        # Giám sát tiến trình Tunnel
        if tunnel_proc and tunnel_proc.poll() is not None:
            print(f"\n[{time.strftime('%H:%M:%S')}] [!] CẢNH BÁO: Tiến trình Tunnel ({tunnel_type}) đã dừng! Đang tự động kết nối lại...")
            if tunnel_type == "Cloudflare":
                tunnel_proc = subprocess.Popen(cf_cmd, stdout=open(cf_log, "a"), stderr=subprocess.STDOUT)
                time.sleep(4)
                new_u = extract_cf_url(cf_log)
                if new_u and new_u != public_url:
                    public_url = new_u
                    print(f"[{time.strftime('%H:%M:%S')}] [✓] Link Cloudflare mới được cấp: {public_url}")
            elif tunnel_type == "Localtunnel":
                tunnel_proc = subprocess.Popen(lt_cmd, stdout=open(lt_log, "a"), stderr=subprocess.STDOUT)
                time.sleep(4)

        # Giám sát tiến trình Server chính
        if server_proc.poll() is not None:
            print(f"\n[{time.strftime('%H:%M:%S')}] [❌] CẢNH BÁO: Server đã dừng đột ngột! Chi tiết log:")
            if os.path.exists(server_log_path):
                with open(server_log_path, "r", errors="ignore") as f:
                    print(f.read()[-1500:])
            print("[*] Đang tự khởi động lại Web Server...")
            server_proc = subprocess.Popen(["npm", "run", "dev"], stdout=open(server_log_path, "a"), stderr=subprocess.STDOUT)
            time.sleep(4)

        # Heartbeat ping mỗi 60 giây để Colab không timeout
        if elapsed - last_heartbeat >= 60:
            last_heartbeat = elapsed
            mins = elapsed // 60
            print(f"[{time.strftime('%H:%M:%S')}] [Heartbeat] Server + Tunnel đang chạy ổn định [{mins} phút] | URL: {public_url}")

except KeyboardInterrupt:
    print("\n" + "=" * 70)
    print("[✓] Đã nhận lệnh ngắt (Stop) từ bạn. Tiến hành tắt Server và giải phóng Tunnel...")
    try:
        tunnel_proc.terminate()
    except Exception:
        pass
    try:
        server_proc.terminate()
    except Exception:
        pass
    print("[✓] Đã dọn dẹp sạch sẽ. Studio đã tắt an toàn!")
    print("=" * 70)
